In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:

import numpy as np
import matplotlib.pyplot as plt
import torch
import random
from src.env import NearFieldBeamFocusingEnv
from src.sac_agent import SACAgent
from src.utils import load_hfss_channel
from src.config import (
    ENV_CONFIG,
    SAC_CONFIG,
    TRAIN_CONFIG,
    HFSS_FILE
)

from src.utils import (
    near_field_channel_to_user,
    near_field_conjugate_weights,
    sph_to_cart
)

import numpy as np
import matplotlib.pyplot as plt

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# Common schedule. 500 episodes of 60 steps = 30,000 interactions in SAC_only.
TOTAL_STEPS = 500 * int(ENV_CONFIG['max_steps'])
LEARNING_STARTS = int(TRAIN_CONFIG['initial_random_steps'])
UPDATES_PER_STEP = int(TRAIN_CONFIG['updates_per_step'])
BATCH_SIZE = int(SAC_CONFIG['batch_size'])
EVAL_EVERY = 5_000
VALIDATION_SEEDS = list(range(10_000, 10_010))
TEST_SEEDS = list(range(20_000, 20_100))
# True reproduces src/train.py's use of terminated OR truncated in replay.
# False uses only genuine termination. Set identically for both agents.
MASK_TIME_LIMITS = True

In [3]:
hsi, selected_freq = load_hfss_channel(
    HFSS_FILE,
    target_freq=ENV_CONFIG["fc"]
)

print("Hsi shape:", hsi.shape)
print("Frequency:", selected_freq / 1e9, "GHz")

Hsi shape: (24, 24)
Frequency: 3.5 GHz


In [4]:
env_config = ENV_CONFIG.copy()
env_config["seed"] = 1234

env = NearFieldBeamFocusingEnv(
    hsi,
    **env_config
)

In [5]:
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)

Observation space: Box(-1.0, 1.0, (19,), float32)
Action space: Box(-1.0, 1.0, (6,), float32)


In [6]:
agent = SACAgent(**SAC_CONFIG)
obs, info = env.reset()

done = False

while not done:

    action = agent.choose_action(obs)

    next_obs, reward, terminated, truncated, info = env.step(action)

    done = terminated or truncated

    agent.remember(
        obs,
        action,
        reward,
        next_obs,
        done
    )

    obs = next_obs

    print(
        f"SI = {info['SI_dB']:.2f} dB, "
        f"reward = {reward:.3f}"
    )

SI = -47.96 dB, reward = 0.678
SI = -48.06 dB, reward = 1.874
SI = -47.79 dB, reward = 0.171
SI = -47.96 dB, reward = 1.509
SI = -48.16 dB, reward = 2.487
SI = -48.02 dB, reward = 1.389
SI = -47.72 dB, reward = -0.220
SI = -47.63 dB, reward = -0.431
SI = -47.65 dB, reward = -0.165
SI = -47.37 dB, reward = -1.868
SI = -47.15 dB, reward = -2.877
SI = -47.01 dB, reward = -3.446
SI = -46.82 dB, reward = -4.394
SI = -46.38 dB, reward = -6.819
SI = -46.12 dB, reward = -7.894
SI = -45.85 dB, reward = -9.161
SI = -45.54 dB, reward = -10.715
SI = -45.42 dB, reward = -11.046
SI = -45.18 dB, reward = -12.381
SI = -44.82 dB, reward = -14.239
SI = -44.50 dB, reward = -15.694
SI = -44.28 dB, reward = -16.678
SI = -43.95 dB, reward = -18.392
SI = -43.68 dB, reward = -19.612
SI = -43.40 dB, reward = -20.000
SI = -43.23 dB, reward = -20.000
SI = -42.88 dB, reward = -20.000
SI = -42.68 dB, reward = -20.000
SI = -42.49 dB, reward = -20.000
SI = -42.32 dB, reward = -20.000
SI = -42.14 dB, reward = -20.000

In [7]:
num_episodes = 500

episode_rewards = []
episode_best_si = []

for episode in range(num_episodes):

    obs, _ = env.reset()

    done = False
    total_reward = 0.0
    best_si = np.inf

    while not done:

        action = agent.choose_action(obs)

        next_obs, reward, terminated, truncated, info = env.step(action)

        done = terminated or truncated

        agent.remember(
            obs,
            action,
            reward,
            next_obs,
            done
        )

        agent.learn()

        obs = next_obs

        total_reward += reward
        best_si = min(best_si, info["SI_dB"])

    episode_rewards.append(total_reward)
    episode_best_si.append(best_si)

    if episode % 10 == 0:
        print(
            f"Episode {episode:4d} | "
            f"reward = {total_reward:8.2f} | "
            f"best SI = {best_si:7.2f} dB"
        )

Episode    0 | reward =   581.37 | best SI =  -27.16 dB
Episode   10 | reward =  -823.42 | best SI =  -38.48 dB
Episode   20 | reward =  -525.52 | best SI =  -44.14 dB
Episode   30 | reward = -1069.32 | best SI =  -37.06 dB
Episode   40 | reward =  -492.02 | best SI =  -25.15 dB
Episode   50 | reward =  -980.19 | best SI =  -34.91 dB
Episode   60 | reward =   -93.44 | best SI =  -31.82 dB
Episode   70 | reward =  -516.79 | best SI =  -42.30 dB
Episode   80 | reward =  -382.20 | best SI =  -36.35 dB
Episode   90 | reward =   252.57 | best SI =  -31.73 dB
Episode  100 | reward =  -419.94 | best SI =  -51.38 dB
Episode  110 | reward =  -244.37 | best SI =  -43.45 dB
Episode  120 | reward =  -397.53 | best SI =  -33.32 dB
Episode  130 | reward =   -61.74 | best SI =  -60.17 dB
Episode  140 | reward =  -350.15 | best SI =  -39.69 dB
Episode  150 | reward =   328.41 | best SI =  -54.03 dB
Episode  160 | reward =  -422.87 | best SI =  -49.74 dB
Episode  170 | reward =  -272.54 | best SI =  -4

Run a deterministic SAC episode and keep the best SI state

In [13]:
obs, _ = env.reset()

done = False

best_si = np.inf
best_info = None

while not done:

    action = agent.choose_action(
        obs,
        evaluate=True
    )

    obs, reward, terminated, truncated, info = env.step(action)

    done = terminated or truncated

    if info["SI_dB"] < best_si:
        best_si = info["SI_dB"]
        best_info = info.copy()

print("Best SI:", best_si)

print(
    "Tx focus:",
    best_info["r_tx_focus"],
    best_info["phi_tx_focus"],
    best_info["theta_tx_focus"]
)

print(
    "Rx focus:",
    best_info["r_rx_focus"],
    best_info["phi_rx_focus"],
    best_info["theta_rx_focus"]
)

Best SI: -23.60524526998354
Tx focus: 2.9050464630126953 -4.871413230895996 8.28242301940918
Rx focus: 2.6871323585510254 4.161082744598389 -13.149967193603516


Construct the final Tx weights

In [15]:
w_tx = near_field_conjugate_weights(
    env.pos_tx,
    r=best_info["r_tx_focus"],
    phi_deg=best_info["phi_tx_focus"],
    theta_deg=best_info["theta_tx_focus"],
    fc=env.fc
)

x_values = np.linspace(0.2, 8.0, 200)
y_values = np.linspace(-5.0, 5.0, 200)

X, Y = np.meshgrid(
    x_values,
    y_values
)

gain = np.zeros_like(X)

for i in range(X.shape[0]):

    for j in range(X.shape[1]):

        x = X[i, j]
        y = Y[i, j]
        z = 0.0

        r = np.sqrt(
            x**2 + y**2 + z**2
        )

        phi = np.degrees(
            np.arctan2(y, x)
        )

        theta = np.degrees(
            np.arctan2(
                z,
                np.sqrt(x**2 + y**2)
            )
        )

        h = near_field_channel_to_user(
            env.pos_tx,
            np.array([
                r,
                theta,
                phi
            ]),
            env.fc
        )

        gain[i, j] = np.abs(
            h @ w_tx
        )**2

gain_db = 10 * np.log10(
    gain / np.max(gain) + 1e-15
)

plt.figure(figsize=(10, 6))

plt.contourf(
    X,
    Y,
    gain_db,
    levels=np.linspace(-30, 0, 61)
)

plt.colorbar(
    label="Normalized Tx gain [dB]"
)

plt.contour(
    X,
    Y,
    gain_db,
    levels=[-3],
    linewidths=2
)

plt.xlabel("x [m]")
plt.ylabel("y [m]")

#plt.plot(x, y, marker='o')

plt.title(
    f"SAC-selected Tx beam\n"
    f"SI = {best_si:.2f} dB"
)

plt.axis("equal")
plt.grid()

plt.show()

/var/folders/9h/65ctz95j25g6fypfgby_mr400000gn/T/ipykernel_19571/3538477279.py:94: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
from src.test_3d_beam_map import (
    rollout_best_focus,
    focus_from_info,
    user_from_case,
    plot_3d_geometry
)

import matplotlib.pyplot as plt

In [17]:
case = {
    "r_dl_user": 3.0,
    "phi_dl_user": 20.0,
    "theta_dl_user": 0.0,

    "r_ul_user": 4.0,
    "phi_ul_user": -25.0,
    "theta_ul_user": 0.0,

    "seed": 1234
}

In [12]:
si_ref_db, best_si_db, best_info, final_info = \
    rollout_best_focus(
        env,
        agent,
        case
    )
print("Reference SI:", si_ref_db)
print("Best SAC SI:", best_si_db)

Reference SI: -35.07408790158311
Best SAC SI: -39.29756675423544


In [18]:
tx_focus = focus_from_info(
    best_info,
    "tx"
)

rx_focus = focus_from_info(
    best_info,
    "rx"
)

print("Tx focus:", tx_focus)
print("Rx focus:", rx_focus)

Tx focus: {'r': 2.9050464630126953, 'phi': -4.871413230895996, 'theta': 8.28242301940918}
Rx focus: {'r': 2.6871323585510254, 'phi': 4.161082744598389, 'theta': -13.149967193603516}


In [19]:
dl_user = user_from_case(
    case,
    "dl"
)

ul_user = user_from_case(
    case,
    "ul"
)

In [22]:
fig = plt.figure(
    figsize=(10, 8)
)

ax = fig.add_subplot(
    111,
    projection="3d"
)

plot_3d_geometry(
    ax=ax,
    env=env,
    tx_user=dl_user,
    rx_user=ul_user,
    tx_focus=tx_focus,
    rx_focus=rx_focus,
    resolution=64,
    title=(
        f"SAC Beam Focusing\n"
        f"Reference SI = {si_ref_db:.2f} dB | "
        f"Best SI = {best_si_db:.2f} dB"
    )
)

plt.savefig("output.png")